# LeetCode 2367 — Number of Arithmetic Triplets bằng Python

Link đề: https://leetcode.com/problems/number-of-arithmetic-triplets/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, Two Pointers, Enumeration

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def arithmeticTriplets(self, nums: List[int], diff: int) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu **bộ ba cấp số cộng** với công sai cố định `diff`, và điều kiện chỉ số `i < j < k` gắn với mảng **tăng ngặt**.
2. Biết liệt kê bộ ba bằng ba vòng lặp lồng nhau, rồi **giảm xuống hai vòng**, rồi **một vòng** bằng `set` để kiểm tra "có tồn tại không" trong `O(1)`.
3. Biết dùng `set`, toán tử `in`, `itertools.combinations`, hai con trỏ (two pointers) trên mảng đã sắp xếp và mảng đánh dấu theo giá trị.
4. Viết 5 cách: ba vòng lặp, hai vòng + `set`, một vòng + `set`, hai con trỏ, và mảng đánh dấu theo giá trị.
5. Trace, test ngẫu nhiên đối chiếu, và tránh lỗi: đếm hoán vị thay vì tổ hợp, quên điều kiện `i < j < k`, kiểm tra `x + diff` mà quên `x + 2 * diff`, đếm trùng khi mảng có phần tử lặp.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` **tăng ngặt** (strictly increasing) và một số nguyên dương `diff`. Một bộ ba `(i, j, k)` là **bộ ba cấp số cộng** nếu thỏa **cả ba**:

- `i < j < k`;
- `nums[j] - nums[i] == diff`;
- `nums[k] - nums[j] == diff`.

Trả về **số lượng** bộ ba cấp số cộng khác nhau.

| Ví dụ | nums | diff | Output | Giải thích |
|---|---|---|---|---|
| 1 | `[0,1,4,6,7,10]` | `3` | `2` | các bộ giá trị `(1,4,7)` và `(4,7,10)` |
| 2 | `[4,5,6,7,8,9]` | `2` | `2` | các bộ giá trị `(4,6,8)` và `(5,7,9)` |

Ở ví dụ 1, hai bộ ba theo **giá trị** là `(1, 4, 7)` và `(4, 7, 10)` (chỉ số `(1,2,4)` và `(2,4,5)`).

Constraints: `3 <= nums.length <= 200`; `0 <= nums[i] <= 200`; `1 <= diff <= 50`; `nums` tăng ngặt.

In [1]:
examples = [([0, 1, 4, 6, 7, 10], 3, 2), ([4, 5, 6, 7, 8, 9], 2, 2)]
for nums, diff, expected in examples:
    found = []
    n = len(nums)
    for i in range(n):
        for j in range(i + 1, n):
            for k in range(j + 1, n):
                if nums[j] - nums[i] == diff and nums[k] - nums[j] == diff:
                    found.append((i, j, k, nums[i], nums[j], nums[k]))
    print(f"nums={nums}, diff={diff}: (i,j,k,giá trị)={found} -> {len(found)} (mong đợi {expected})")

nums=[0, 1, 4, 6, 7, 10], diff=3: (i,j,k,giá trị)=[(1, 2, 4, 1, 4, 7), (2, 4, 5, 4, 7, 10)] -> 2 (mong đợi 2)
nums=[4, 5, 6, 7, 8, 9], diff=2: (i,j,k,giá trị)=[(0, 2, 4, 4, 6, 8), (1, 3, 5, 5, 7, 9)] -> 2 (mong đợi 2)


## 3. Kiến thức nền

### 3.1. Cấp số cộng và "tăng ngặt"

Ba số `a, b, c` lập thành cấp số cộng công sai `d` khi `b = a + d` và `c = b + d = a + 2d`. Mảng **tăng ngặt** nghĩa là `nums[0] < nums[1] < ...` — các giá trị **đôi một khác nhau** và đã sắp xếp. Vì `diff > 0` nên `nums[j] = nums[i] + diff > nums[i]`, tức tự nhiên `j > i`: ta không cần kiểm tra `i < j < k` riêng nếu tìm theo **giá trị**.

In [2]:
nums = [0, 1, 4, 6, 7, 10]
print(all(nums[i] < nums[i + 1] for i in range(len(nums) - 1)))     # tăng ngặt
a, d = 1, 3
print(a, a + d, a + 2 * d)                                           # 1 4 7

True
1 4 7


### 3.2. Ba vòng lặp lồng nhau với chỉ số tăng dần

`for i in range(n): for j in range(i+1, n): for k in range(j+1, n):` liệt kê mọi bộ `i < j < k` đúng một lần. Với `n = 200` có `C(200,3) ≈ 1.3 triệu` bộ — vẫn chạy được trong Python nhưng chậm.

In [3]:
from itertools import combinations

n = 5
triples = [(i, j, k) for i in range(n) for j in range(i + 1, n) for k in range(j + 1, n)]
print(len(triples), triples[:4])
print(list(combinations(range(n), 3)) == triples)     # combinations làm cùng việc

10 [(0, 1, 2), (0, 1, 3), (0, 1, 4), (0, 2, 3)]
True


### 3.3. `set` và kiểm tra thành viên `x in s`

`set` băm giá trị, nên `x in s` chạy **`O(1)` trung bình** (so với `O(n)` của `list`). Biến `nums` thành `set` một lần rồi tra cứu rất nhiều lần.

In [4]:
nums = [0, 1, 4, 6, 7, 10]
s = set(nums)
print(4 in s, 5 in s)
print([x for x in nums if x + 3 in s])        # những x có 'x + 3' cũng nằm trong mảng

True False
[1, 4, 7]


### 3.4. Hai con trỏ trên mảng đã sắp xếp

Khi mảng sắp xếp, để tìm cặp có hiệu bằng một số cố định, ta để con trỏ thứ hai **chỉ đi tới** (không quay lại) — mỗi con trỏ đi nhiều nhất `n` bước, tổng `O(n)`. Ở đây mảng đã tăng ngặt nên không phải sắp thêm.

In [5]:
nums = [0, 1, 4, 6, 7, 10]
diff = 3
j = 0
pairs = []
for i in range(len(nums)):
    while j < len(nums) and nums[j] - nums[i] < diff:
        j += 1
    if j < len(nums) and nums[j] - nums[i] == diff:
        pairs.append((nums[i], nums[j]))
print("các cặp (a, a+3):", pairs)

các cặp (a, a+3): [(1, 4), (4, 7), (7, 10)]


### 3.5. Mảng đánh dấu theo giá trị

Giá trị chỉ `0..200`, nên `present = [False] * 201` đánh dấu giá trị có mặt; tra `present[x + diff]` không cần băm. Chú ý chặn chỉ số: `x + 2 * diff` có thể vượt `200`.

In [6]:
nums = [0, 1, 4, 6, 7, 10]
present = [False] * 201
for v in nums:
    present[v] = True
print([v for v in nums if v + 6 <= 200 and present[v + 3] and present[v + 6]])

[1, 4]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Với `nums = [0,1,4,6,7,10]`, `diff = 3`, hãy xét từng số `x` làm **phần tử đầu** của bộ ba:

- `x = 0`: cần `3` và `6`. Có `6` nhưng không có `3` -> không.
- `x = 1`: cần `4` và `7`. Có cả hai -> **bộ ba (1, 4, 7)**.
- `x = 4`: cần `7` và `10`. Có cả hai -> **bộ ba (4, 7, 10)**.
- `x = 6`: cần `9` và `12`. Không -> không.
- `x = 7`: cần `10` và `13`. Có `10` nhưng không có `13` -> không.
- `x = 10`: cần `13`, `16` -> không.

Tổng `2`. Khóa trực giác: một bộ ba hoàn toàn được xác định bởi **phần tử đầu `x`** (giá trị `x`, `x+diff`, `x+2diff`). Mảng tăng ngặt nên mỗi giá trị chỉ xuất hiện một lần, vì vậy "số bộ ba" = "số giá trị `x` trong mảng sao cho `x + diff` và `x + 2*diff` cũng trong mảng".

In [7]:
nums = [0, 1, 4, 6, 7, 10]
diff = 3
s = set(nums)
for x in nums:
    ok = (x + diff in s) and (x + 2 * diff in s)
    print(f"x={x:2}: cần {x + diff:2} và {x + 2 * diff:2} -> {'CÓ bộ ba' if ok else 'không'}")

x= 0: cần  3 và  6 -> không
x= 1: cần  4 và  7 -> CÓ bộ ba
x= 4: cần  7 và 10 -> CÓ bộ ba
x= 6: cần  9 và 12 -> không
x= 7: cần 10 và 13 -> không
x=10: cần 13 và 16 -> không


## 5. Cách 1 — Brute Force / baseline: ba vòng lặp

**Ý tưởng:** liệt kê mọi `i < j < k`, kiểm tra hai hiệu cùng bằng `diff`, đếm.

**Độ phức tạp:** `O(n^3)`; với `n = 200` khoảng `1,3 triệu` bộ — chạy được nhưng lãng phí. **Nút thắt:** sau khi chọn `i`, `j`, giá trị cần cho `k` là **duy nhất** (`nums[j] + diff`), không cần thử mọi `k`.

In [8]:
from typing import List

class SolutionTriple:
    def arithmeticTriplets(self, nums: List[int], diff: int) -> int:
        n, count = len(nums), 0
        for i in range(n):
            for j in range(i + 1, n):
                for k in range(j + 1, n):
                    if nums[j] - nums[i] == diff and nums[k] - nums[j] == diff:
                        count += 1
        return count

for arr, d in (([0, 1, 4, 6, 7, 10], 3), ([4, 5, 6, 7, 8, 9], 2)):
    print(arr, d, "->", SolutionTriple().arithmeticTriplets(arr, d))

[0, 1, 4, 6, 7, 10] 3 -> 2
[4, 5, 6, 7, 8, 9] 2 -> 2


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1:** bộ ba được xác định bởi giá trị đầu `x`: ba giá trị là `x`, `x + diff`, `x + 2*diff`. Vì `diff > 0` và mảng tăng ngặt, các **chỉ số** của ba giá trị đó tự động tăng dần (`i < j < k`), nên chỉ cần kiểm tra **sự tồn tại** của ba giá trị.

**Insight 2:** kiểm tra tồn tại trong `O(1)` bằng `set` (hoặc mảng đánh dấu), không cần vòng lặp thứ ba (hay thứ hai).

**Invariant:** mỗi giá trị `x` trong `nums` được xét đúng một lần và đóng góp `1` nếu `x + diff` và `x + 2*diff` đều có mặt, `0` nếu ngược lại. Hai giá trị `x` khác nhau cho hai bộ ba khác nhau (phần tử đầu khác nhau).

**Quy tắc:** `answer = số x trong nums thỏa x + diff in S và x + 2*diff in S`, với `S = set(nums)`.

## 7. Các cách tiếp theo

### Cách 2 — Hai vòng lặp + `set` cho phần tử thứ ba

Chọn `i < j` với `nums[j] - nums[i] == diff`, rồi kiểm tra `nums[j] + diff in s` (giá trị của `k`).

In [9]:
from typing import List

class SolutionTwoLoops:
    def arithmeticTriplets(self, nums: List[int], diff: int) -> int:
        s = set(nums)
        n, count = len(nums), 0
        for i in range(n):
            for j in range(i + 1, n):
                if nums[j] - nums[i] == diff and nums[j] + diff in s:
                    count += 1
        return count

for arr, d in (([0, 1, 4, 6, 7, 10], 3), ([4, 5, 6, 7, 8, 9], 2)):
    print(arr, d, "->", SolutionTwoLoops().arithmeticTriplets(arr, d))

[0, 1, 4, 6, 7, 10] 3 -> 2
[4, 5, 6, 7, 8, 9] 2 -> 2


**Độ phức tạp Cách 2:** Time `O(n^2)`, Space `O(n)`. Vì giá trị `nums[j] + diff` là số lớn hơn `nums[j]`, nó chỉ có thể nằm ở chỉ số `k > j` (mảng tăng ngặt), nên điều kiện `k > j` tự thỏa.

### Cách 3 — Một vòng lặp + `set` (cách chuẩn)

Với mỗi `x`, kiểm tra `x + diff` và `x + 2*diff` có trong `set`. `O(n)`.

In [10]:
from typing import List

class SolutionSet:
    def arithmeticTriplets(self, nums: List[int], diff: int) -> int:
        s = set(nums)
        return sum(1 for x in nums if x + diff in s and x + 2 * diff in s)

for arr, d in (([0, 1, 4, 6, 7, 10], 3), ([4, 5, 6, 7, 8, 9], 2)):
    print(arr, d, "->", SolutionSet().arithmeticTriplets(arr, d))

[0, 1, 4, 6, 7, 10] 3 -> 2
[4, 5, 6, 7, 8, 9] 2 -> 2


**Độ phức tạp Cách 3:** Time `O(n)`, Space `O(n)`.

### Cách 4 — Hai con trỏ (không dùng bộ nhớ phụ)

Mảng đã sắp. Với mỗi `i`, tìm `j` mà `nums[j] == nums[i] + diff` bằng con trỏ chỉ đi tới, rồi tìm `k` mà `nums[k] == nums[j] + diff` bằng con trỏ thứ hai cũng chỉ đi tới. Vì `nums[i]` tăng nên các mục tiêu cũng tăng, con trỏ không cần lùi.

In [11]:
from typing import List

class SolutionTwoPointers:
    def arithmeticTriplets(self, nums: List[int], diff: int) -> int:
        n, count = len(nums), 0
        j = k = 0
        for i in range(n):
            while j < n and nums[j] < nums[i] + diff:
                j += 1
            while k < n and nums[k] < nums[i] + 2 * diff:
                k += 1
            if j < n and k < n and nums[j] == nums[i] + diff and nums[k] == nums[i] + 2 * diff:
                count += 1
        return count

for arr, d in (([0, 1, 4, 6, 7, 10], 3), ([4, 5, 6, 7, 8, 9], 2)):
    print(arr, d, "->", SolutionTwoPointers().arithmeticTriplets(arr, d))

[0, 1, 4, 6, 7, 10] 3 -> 2
[4, 5, 6, 7, 8, 9] 2 -> 2


**Độ phức tạp Cách 4:** Time `O(n)` (mỗi con trỏ đi tối đa `n` bước), Space `O(1)`. Cách này tận dụng tính **đã sắp xếp** và là cách tiết kiệm bộ nhớ nhất.

### Cách 5 — Mảng đánh dấu theo giá trị

Vì `nums[i] <= 200`, dùng `present = [False] * 201`. Nhớ chặn `x + 2 * diff <= 200`.

In [12]:
from typing import List

class SolutionMarks:
    def arithmeticTriplets(self, nums: List[int], diff: int) -> int:
        top = max(nums)
        present = [False] * (top + 1)
        for v in nums:
            present[v] = True
        count = 0
        for x in nums:
            if x + 2 * diff <= top and present[x + diff] and present[x + 2 * diff]:
                count += 1
        return count

for arr, d in (([0, 1, 4, 6, 7, 10], 3), ([4, 5, 6, 7, 8, 9], 2), ([1, 2, 3], 50)):
    print(arr, d, "->", SolutionMarks().arithmeticTriplets(arr, d))

[0, 1, 4, 6, 7, 10] 3 -> 2
[4, 5, 6, 7, 8, 9] 2 -> 2
[1, 2, 3] 50 -> 0


**Độ phức tạp Cách 5:** Time `O(n + max)`, Space `O(max)` = `O(201)`. Chặn `x + 2*diff <= top` ngăn lỗi chỉ số ngoài mảng.

## 8. TRACE từng bước

Trace Cách 3 với `nums = [0,1,4,6,7,10]`, `diff = 3`, `S = {0,1,4,6,7,10}`:

| `x` | `x + 3` | trong `S`? | `x + 6` | trong `S`? | cộng |
|---|---|---|---|---|---|
| 0 | 3 | không | 6 | có | 0 |
| 1 | 4 | có | 7 | có | **1** |
| 4 | 7 | có | 10 | có | **1** |
| 6 | 9 | không | 12 | không | 0 |
| 7 | 10 | có | 13 | không | 0 |
| 10 | 13 | không | 16 | không | 0 |

Tổng `2`.

In [13]:
def trace_set(nums, diff):
    s = set(nums)
    total = 0
    print(f"S={sorted(s)}, diff={diff}")
    for x in nums:
        a, b = x + diff, x + 2 * diff
        ok = a in s and b in s
        total += ok
        print(f"  x={x:3}: x+diff={a:3} ({'có' if a in s else 'không'}), x+2diff={b:3} ({'có' if b in s else 'không'}) -> {'+1' if ok else '+0'}  tổng={total}")
    return total

print("kết quả:", trace_set([0, 1, 4, 6, 7, 10], 3))
print("kết quả:", trace_set([4, 5, 6, 7, 8, 9], 2))

S=[0, 1, 4, 6, 7, 10], diff=3
  x=  0: x+diff=  3 (không), x+2diff=  6 (có) -> +0  tổng=0
  x=  1: x+diff=  4 (có), x+2diff=  7 (có) -> +1  tổng=1
  x=  4: x+diff=  7 (có), x+2diff= 10 (có) -> +1  tổng=2
  x=  6: x+diff=  9 (không), x+2diff= 12 (không) -> +0  tổng=2
  x=  7: x+diff= 10 (có), x+2diff= 13 (không) -> +0  tổng=2
  x= 10: x+diff= 13 (không), x+2diff= 16 (không) -> +0  tổng=2
kết quả: 2
S=[4, 5, 6, 7, 8, 9], diff=2
  x=  4: x+diff=  6 (có), x+2diff=  8 (có) -> +1  tổng=1
  x=  5: x+diff=  7 (có), x+2diff=  9 (có) -> +1  tổng=2
  x=  6: x+diff=  8 (có), x+2diff= 10 (không) -> +0  tổng=2
  x=  7: x+diff=  9 (có), x+2diff= 11 (không) -> +0  tổng=2
  x=  8: x+diff= 10 (không), x+2diff= 12 (không) -> +0  tổng=2
  x=  9: x+diff= 11 (không), x+2diff= 13 (không) -> +0  tổng=2
kết quả: 2


In [14]:
def trace_two_pointers(nums, diff):
    n, j, k, count = len(nums), 0, 0, 0
    for i in range(n):
        while j < n and nums[j] < nums[i] + diff:
            j += 1
        while k < n and nums[k] < nums[i] + 2 * diff:
            k += 1
        hit = j < n and k < n and nums[j] == nums[i] + diff and nums[k] == nums[i] + 2 * diff
        count += hit
        print(f"  i={i} (x={nums[i]}): j={j} (giá trị {nums[j] if j < n else None}), k={k} (giá trị {nums[k] if k < n else None}) -> {'+1' if hit else '+0'} count={count}")
    return count

print("kết quả:", trace_two_pointers([0, 1, 4, 6, 7, 10], 3))

  i=0 (x=0): j=2 (giá trị 4), k=3 (giá trị 6) -> +0 count=0
  i=1 (x=1): j=2 (giá trị 4), k=4 (giá trị 7) -> +1 count=1
  i=2 (x=4): j=4 (giá trị 7), k=5 (giá trị 10) -> +1 count=2
  i=3 (x=6): j=5 (giá trị 10), k=6 (giá trị None) -> +0 count=2
  i=4 (x=7): j=5 (giá trị 10), k=6 (giá trị None) -> +0 count=2
  i=5 (x=10): j=6 (giá trị None), k=6 (giá trị None) -> +0 count=2
kết quả: 2


## 9. Code nộp LeetCode

In [15]:
from typing import List

class Solution:
    def arithmeticTriplets(self, nums: List[int], diff: int) -> int:
        s = set(nums)
        return sum(1 for x in nums if x + diff in s and x + 2 * diff in s)

print(Solution().arithmeticTriplets([0, 1, 4, 6, 7, 10], 3))
print(Solution().arithmeticTriplets([4, 5, 6, 7, 8, 9], 2))

2
2


## 10. Vì sao thuật toán đúng

- **Bộ ba <-> giá trị đầu.** Một bộ ba `(i, j, k)` thỏa đề cho ba giá trị `a = nums[i]`, `a + diff`, `a + 2*diff`. Ngược lại nếu ba giá trị này đều có mặt trong `nums` thì có đúng một bộ chỉ số tương ứng (mảng tăng ngặt nên mỗi giá trị có đúng một chỉ số).
- **Thứ tự chỉ số tự thỏa.** `diff > 0` nên `a < a + diff < a + 2*diff`; mảng tăng ngặt, nên chỉ số của giá trị lớn hơn cũng lớn hơn: `i < j < k`.
- **Không đếm trùng.** Hai bộ ba khác nhau có phần tử đầu `a` khác nhau (nếu cùng `a` thì cùng `(a, a+diff, a+2diff)`), và mỗi `a` được duyệt một lần.
- **Cách 4 (hai con trỏ).** Khi `nums[i]` tăng, `nums[i] + diff` cũng tăng nên chỉ số đầu tiên có giá trị `>= nums[i] + diff` không bao giờ giảm — con trỏ `j` (và `k`) chỉ đi tới là đủ.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)` (`<= 200`), `M = max(nums)` (`<= 200`).

| Cách | Time | Space |
|---|---|---|
| 1. Ba vòng lặp | `O(n^3)` | `O(1)` |
| 2. Hai vòng + `set` | `O(n^2)` | `O(n)` |
| 3. Một vòng + `set` | `O(n)` | `O(n)` |
| 4. Hai con trỏ | `O(n)` | `O(1)` |
| 5. Mảng đánh dấu | `O(n + M)` | `O(M)` |

Với `n <= 200`, kể cả `O(n^3)` vẫn qua (khoảng 1,3 triệu bước), nhưng cách `O(n)` là chuẩn mực để học: đổi "tìm kiếm" thành "tra cứu `set`".

## 12. Bộ test

Đối chiếu tất cả các cách trên ví dụ đề, edge case và test ngẫu nhiên có `seed`.

In [16]:
import random

solutions = [SolutionTriple(), SolutionTwoLoops(), SolutionSet(), SolutionTwoPointers(), SolutionMarks(), Solution()]

fixed = [
    ([0, 1, 4, 6, 7, 10], 3, 2),
    ([4, 5, 6, 7, 8, 9], 2, 2),
    ([0, 1, 2], 1, 1),                  # đúng 3 phần tử tạo bộ ba
    ([0, 1, 2], 2, 0),                  # công sai quá lớn
    ([1, 2, 3, 4, 5], 1, 3),            # (1,2,3),(2,3,4),(3,4,5)
    ([0, 50, 100, 150, 200], 50, 3),    # biên: giá trị tới 200
    ([0, 10, 20, 30, 40, 50], 10, 4),
    ([1, 3, 5, 7, 9, 11], 4, 2),        # (1,5,9) và (3,7,11)
    ([0, 3, 7], 3, 0),                  # thiếu phần tử thứ ba
    ([5, 6, 7], 50, 0),                 # diff lớn hơn mọi khoảng cách
]
for nums, diff, expected in fixed:
    for s in solutions:
        assert s.arithmeticTriplets(list(nums), diff) == expected, (type(s).__name__, nums, diff, expected)
print("đã qua", len(fixed), "test cố định cho", len(solutions), "cách")

đã qua 10 test cố định cho 6 cách


In [17]:
random.seed(2367)
for _ in range(500):
    n = random.randint(3, 40)
    nums = sorted(random.sample(range(0, 201), n))      # tăng ngặt
    diff = random.randint(1, 50)
    ref = SolutionTriple().arithmeticTriplets(list(nums), diff)
    for s in solutions[1:]:
        assert s.arithmeticTriplets(list(nums), diff) == ref, (type(s).__name__, nums, diff)
print("500 test ngẫu nhiên đối chiếu: OK")

500 test ngẫu nhiên đối chiếu: OK


In [18]:
# mảng lớn nhất theo đề: n = 200, giá trị 0..199 và vài công sai; chỉ so các cách nhanh với brute (1.3 triệu bộ)
nums = list(range(200))
for diff in (1, 7, 50):
    ref = SolutionTriple().arithmeticTriplets(list(nums), diff)
    assert ref == max(0, 200 - 2 * diff)
    for s in solutions[1:]:
        assert s.arithmeticTriplets(list(nums), diff) == ref
print("n = 200: các cách khớp, và đáp án = 200 - 2*diff")

n = 200: các cách khớp, và đáp án = 200 - 2*diff


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chỉ kiểm tra `x + diff`, quên `x + 2 * diff`

Hai số `x`, `x + diff` mới chỉ là một **cặp**, chưa phải bộ ba. Đếm như vậy cho kết quả lớn hơn đáp án.

In [19]:
def sai_chi_mot_buoc(nums, diff):
    s = set(nums)
    return sum(1 for x in nums if x + diff in s)        # SAI: thiếu x + 2*diff

print("sai :", sai_chi_mot_buoc([0, 1, 4, 6, 7, 10], 3), "| đúng:", Solution().arithmeticTriplets([0, 1, 4, 6, 7, 10], 3))

sai : 3 | đúng: 2


### Lỗi 2: Kiểm tra `x - diff` và `x + diff` rồi đếm mỗi bộ ba nhiều lần

Nếu lấy `x` làm phần tử **giữa** (kiểm tra `x - diff` và `x + diff`), vẫn đúng. Nhưng nếu kiểm tra **cả ba** vai trò rồi cộng lại thì mỗi bộ ba bị đếm 3 lần. Phải chọn **một** vai trò và dùng nhất quán.

In [20]:
def sai_dem_ba_lan(nums, diff):
    s = set(nums)
    total = 0
    for x in nums:
        total += (x + diff in s and x + 2 * diff in s)       # x là phần tử đầu
        total += (x - diff in s and x + diff in s)           # x là phần tử giữa
        total += (x - diff in s and x - 2 * diff in s)       # x là phần tử cuối
    return total

def dung_vai_tro_giua(nums, diff):
    s = set(nums)
    return sum(1 for x in nums if x - diff in s and x + diff in s)

print("sai :", sai_dem_ba_lan([0, 1, 4, 6, 7, 10], 3))
print("đúng (x là phần tử giữa):", dung_vai_tro_giua([0, 1, 4, 6, 7, 10], 3), "| Solution:", Solution().arithmeticTriplets([0, 1, 4, 6, 7, 10], 3))

sai : 6
đúng (x là phần tử giữa): 2 | Solution: 2


### Lỗi 3: Dùng `list` thay `set` để kiểm tra thành viên (chậm)

`x + diff in nums` với `nums` là list quét `O(n)` mỗi lần, làm cả bài `O(n^2)`. Kết quả vẫn đúng nhưng chậm; với `n = 10^5` sẽ quá thời gian. Hãy đổi sang `set` hoặc dùng hai con trỏ.

In [21]:
import time

big = list(range(0, 4000))
t0 = time.perf_counter()
a = sum(1 for x in big if x + 1 in big and x + 2 in big)         # CHẬM: kiểm tra trong list
t1 = time.perf_counter()
s = set(big)
b = sum(1 for x in big if x + 1 in s and x + 2 in s)             # NHANH: kiểm tra trong set
t2 = time.perf_counter()
print("kết quả bằng nhau:", a == b, "| list chậm hơn set (đúng):", (t1 - t0) > (t2 - t1))

kết quả bằng nhau: True | list chậm hơn set (đúng): True


### Lỗi 4: Quên chặn chỉ số ở mảng đánh dấu

Với `present = [False] * 201`, `present[x + 2 * diff]` có thể vượt chỉ số `200` (`IndexError`). Phải kiểm tra `x + 2 * diff <= top` trước.

In [22]:
present = [False] * 201
present[190] = True
x, diff = 190, 50
try:
    print(present[x + diff])
except IndexError as e:
    print("IndexError:", e)
print("đúng (có chặn chỉ số):", SolutionMarks().arithmeticTriplets([0, 50, 100, 190], 50))

IndexError: list index out of range
đúng (có chặn chỉ số): 1


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đếm bộ ba/bộ đôi thỏa hiệu cố định" -> một phần tử **xác định** các phần tử còn lại; dùng `set` tra cứu thay vòng lặp lồng.
- Mảng **tăng ngặt** nghĩa là giá trị đôi một khác nhau và đã sắp -> hai con trỏ không cần sắp thêm.
- "Tồn tại hay không" -> `in set`.
- `n <= 200` cho phép `O(n^3)`; nhưng luôn thử giảm một cấp lặp bằng `set` để quen tay.
- Bài cùng pattern: Two Sum, K-diff Pairs in an Array, Longest Arithmetic Subsequence, Count Number of Pairs With Absolute Difference K.

## 15. Biến thể của bài

- **Mảng không tăng ngặt, có phần tử lặp:** phải đếm số cách chọn chỉ số (dùng `Counter` và nhân số lần).
- **Cấp số cộng độ dài `L` bất kỳ**: lặp `x, x+diff, ..., x+(L-1)*diff` đều trong `set`.
- **Cấp số cộng công sai tùy ý** (Arithmetic Slices): chọn cặp `(i, j)` làm hai số đầu rồi đếm; dùng DP.
- **Cấp số nhân** (`nums[j] = nums[i] * r`): thay `+ diff` bằng `* r`, nhớ xét `0`.
- **Đếm chỉ số `(i, j, k)` với hiệu bằng nhau nhưng không cố định `diff`**: đếm theo `nums[j]` làm phần tử giữa, nhân số cặp đối xứng.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Ba vòng lặp | mọi `i < j < k` | `O(n^3)` | `O(1)` | `n` nhỏ, đối chiếu |
| 2. Hai vòng + `set` | `set` cho phần tử thứ ba | `O(n^2)` | `O(n)` | bước trung gian học |
| 3. Một vòng + `set` | kiểm `x+diff`, `x+2diff` | `O(n)` | `O(n)` | nộp bài, chuẩn |
| 4. Hai con trỏ | tận dụng mảng đã sắp | `O(n)` | `O(1)` | không muốn dùng bộ nhớ phụ |
| 5. Mảng đánh dấu | `present[v]` | `O(n + M)` | `O(M)` | giá trị nhỏ cố định |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, dự đoán `Solution().arithmeticTriplets([1, 2, 3, 4, 5, 6], 2)`.

**Bài 2 (tính tay).** Với `nums = [2, 5, 8, 11, 14]`, `diff = 3`, liệt kê các bộ ba theo giá trị.

**Bài 3 (điền TODO).** Hoàn thành `has_next_two(x, s, diff)` trả `True` nếu `x + diff` và `x + 2*diff` đều trong tập `s` (cell dưới chưa được gọi).

**Bài 4 (tự viết lại).** Viết `list_triplets(nums, diff)` trả về **danh sách** các bộ ba giá trị `(a, a+diff, a+2*diff)` theo thứ tự tăng dần.

**Bài 5 (mini challenge).** Viết `count_length_L(nums, diff, L)` đếm số cấp số cộng **độ dài `L`** (cho `L >= 2`) công sai `diff` trong mảng tăng ngặt.

In [23]:
# TODO Bài 3: điền vào chỗ trống, KHÔNG gọi hàm trong cell này
def has_next_two_todo(x, s, diff):
    # return (x + ?) in s and (x + ?) in s
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: các bộ `(1,3,5)`, `(2,4,6)` -> 2. Bài 2: `(2,5,8)`, `(5,8,11)`, `(8,11,14)`.
- **Mức 2 (vừa):** Bài 3: `x + diff in s and x + 2 * diff in s`. Bài 4: duyệt `x` theo thứ tự mảng, giữ bộ ba nếu `has_next_two`.
- **Mức 3 (sâu):** Bài 5: với mỗi `x`, kiểm tra `all(x + t * diff in s for t in range(L))`. Khi `L = 3` phải khớp `Solution`; khi `L = 2` là đếm cặp `x, x + diff`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [24]:
# Bài 1 và 2
assert Solution().arithmeticTriplets([1, 2, 3, 4, 5, 6], 2) == 2
assert Solution().arithmeticTriplets([2, 5, 8, 11, 14], 3) == 3

# Bài 3
def has_next_two(x, s, diff):
    return (x + diff) in s and (x + 2 * diff) in s

# Bài 4
def list_triplets(nums, diff):
    s = set(nums)
    return [(x, x + diff, x + 2 * diff) for x in nums if has_next_two(x, s, diff)]

# Bài 5
def count_length_L(nums, diff, L):
    s = set(nums)
    return sum(1 for x in nums if all(x + t * diff in s for t in range(L)))

assert has_next_two(1, {1, 4, 7}, 3) and not has_next_two(4, {1, 4, 7}, 3)
assert list_triplets([2, 5, 8, 11, 14], 3) == [(2, 5, 8), (5, 8, 11), (8, 11, 14)]
assert list_triplets([0, 1, 4, 6, 7, 10], 3) == [(1, 4, 7), (4, 7, 10)]
assert count_length_L([0, 1, 4, 6, 7, 10], 3, 3) == Solution().arithmeticTriplets([0, 1, 4, 6, 7, 10], 3)
assert count_length_L([0, 1, 4, 6, 7, 10], 3, 2) == 3      # các cặp (1,4), (4,7), (7,10); không có 3 nên (0,3) bị loại
print("tất cả bài tập OK")

tất cả bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đổi `x + diff`, `x + 2 * diff` thành tra cứu `set`)

```python
class Solution:
    def arithmeticTriplets(self, nums: List[int], diff: int) -> int:
        s = set(nums)
        return sum(1 for x in nums if x + diff in s and x + 2 * diff in s)
```

**Checklist**

- [ ] Hiểu một bộ ba hoàn toàn xác định bởi phần tử đầu `x`?
- [ ] Kiểm tra **cả hai** `x + diff` và `x + 2 * diff`?
- [ ] Mỗi bộ ba chỉ đếm **một lần** (một vai trò cố định cho `x`)?
- [ ] Dùng `set` (hoặc hai con trỏ), không dùng `x in list` trong vòng lặp?
- [ ] Nếu dùng mảng đánh dấu: chặn `x + 2 * diff <= max(nums)`?
- [ ] Đã thử `[0,1,2]` với `diff = 1` (đúng 1), `diff` rất lớn (0), và `n = 200`?
- [ ] Signature đúng `arithmeticTriplets(self, nums: List[int], diff: int) -> int` và có `from typing import List`?